# 牙齿三维网格补底

In [1]:
# 导入必要的包以及定义函数
import os
import glob
import base64
import time
import requests
import json
import trimesh
import urllib
import numpy as np

def _create_colors():
    # 20 high contrast colors
    colors = [[230, 25, 75,255],[60, 180, 75,255],[255, 225, 25,255],\
            [0, 130, 200,255],[245, 130, 48,255],[145, 30, 180,255],[70, 240, 240,255],\
            [240, 50, 230,255],[210, 245, 60,255],[250, 190, 190,255],[0, 128, 128,255],\
            [230, 190, 255,255],[170, 110, 40,255],[255, 250, 200,255],[128, 0, 0,255],\
            [170, 255, 195,255],[128, 128, 0, 255]]
    #np.random.shuffle(colors)
    # gum color
    colors = [[255,255,255,255]] + colors
    return colors

def colored_mesh(mesh, label):
    COLORS = _create_colors()
    mcopy = mesh.copy()
    for i, l in enumerate(np.unique(label)):
        mcopy.visual.face_colors[np.where(label == l)[0]] = COLORS[i % 18]
    return mcopy

# 定义调用规则

请根据您从我方获取的信息修改以下代码块

In [2]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


In [3]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

## 牙齿邻面补全
该接口对传入的半颌口扫或硅胶模型进行预处理、牙齿分割、牙号预测、牙轴预测、邻接面补全与FA点预测。  https://www.chohotech.com/docs/cloud-zh/#/workflow/oral-comp-and-axis-1

In [4]:
json_call = {
  "spec_group": "mesh-processing", # 调用的工作流组， 随API文档发送
  "spec_name": "oral-comp-and-axis", # 调用的工作流名称， 随API文档发送, 
  "spec_version": "1.0-snapshot", # 调用的工作流版本，随API文档发送
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "mesh": {"type":"ply", "data": upload_file('lower_jaw_scan.ply')},
      "jaw_type": 'Lower'
  },
  "output_config": {
      "mesh": {"type": "ply"},
      "teeth": {"type": "ply"},
      "teeth_comp": {"type": "ply"}
  }
}
result = run_job_and_get_results(json_call, 300)

workflow id is wf_1766494919-1b3a6b58-bdca-4db4-8f6f-8da64321e920
API finished in 37.8234806060791s


In [5]:
print('牙号列表：', np.unique(result['seg_labels']))
colored_mesh(retrieve_mesh(result['mesh']), result['seg_labels']).show()

牙号列表： [ 0 31 32 33 34 35 36 37 42 43 44 45 46 47]


In [6]:
# 获取所有牙齿编号
all_tooth_ids = list(result['teeth_comp'].keys())
print(f"\n所有可处理的牙齿: {all_tooth_ids}")


所有可处理的牙齿: ['31', '32', '33', '34', '35', '36', '37', '42', '43', '44', '45', '46', '47']


## 牙齿三维网格补底

teeth 为补全邻接面后的牙齿三维模型，该工作流用于简化close-tooth-bottom功能模块的调用 https://www.chohotech.com/docs/cloud-zh/#/workflow/close-teeth-bottom-1

In [7]:
json_call_close = {
  "spec_group": "mesh-processing",
  "spec_name": "close-teeth-bottom",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "teeth": result['teeth_comp']
  },
  "output_config": {
      "teeth": {"type": "ply"}  # 明确指定输出类型
  }
}
result_close = run_job_and_get_results(json_call_close, 300)

workflow id is wf_1766494959-cd8eece5-1ba8-4f95-92d4-984a3f51073c
API finished in 53.69943356513977s


In [8]:
print(f"\nclose-teeth-bottom 输出中包含 {len(result_close['teeth'])} 个牙齿")
print(f"牙齿编号: {list(result_close['teeth'].keys())}")


close-teeth-bottom 输出中包含 13 个牙齿
牙齿编号: ['43', '42', '35', '37', '45', '32', '46', '33', '44', '31', '36', '34', '47']


In [12]:
closed = retrieve_mesh(result_close['teeth']['37'])
closed.visual.face_colors = [255, 0, 0, 255]  # 红色

original = retrieve_mesh(result['teeth_comp']['37'])
original.visual.face_colors = [150, 150, 150, 255]  # 灰色

print(f"37号牙补底:")
(closed + original).show()  # 合并显示

37号牙补底:
